In [28]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [29]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [30]:
block_size = 8
batch_size = 4

In [31]:
with open('../data/wizard_of_oz.txt', 'r', encoding='utf-8')as f:
    text = f.read()

chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [32]:
strng_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_strng = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [strng_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_strng[i] for i in l])

# encoded_hello = torch.tensor(encode('hello'),dtype=torch.long)
# decoded_hello = decode(encoded_hello.tolist())

data = torch.tensor(encode(text), dtype=torch.long)

In [33]:
n = int(0.8*len(data))

train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    x= torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

tensor([ 87621,  48605, 112603,  69413])
inputs:
tensor([[57,  9,  1, 54, 67, 57,  1, 54],
        [68, 74, 67, 57,  1, 62, 67,  1],
        [61, 73,  1, 72, 68,  9,  3,  1],
        [71, 56, 58, 71, 62, 58, 72,  1]])
targets:
tensor([[ 9,  1, 54, 67, 57,  1, 54, 73],
        [74, 67, 57,  1, 62, 67,  1, 73],
        [73,  1, 72, 68,  9,  3,  1, 72],
        [56, 58, 71, 62, 58, 72,  1, 78]])


In [34]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('When input is:',context,'target is:',target)

When input is: tensor([28]) target is: tensor(39)
When input is: tensor([28, 39]) target is: tensor(42)
When input is: tensor([28, 39, 42]) target is: tensor(39)
When input is: tensor([28, 39, 42, 39]) target is: tensor(44)
When input is: tensor([28, 39, 42, 39, 44]) target is: tensor(32)
When input is: tensor([28, 39, 42, 39, 44, 32]) target is: tensor(49)
When input is: tensor([28, 39, 42, 39, 44, 32, 49]) target is: tensor(1)
When input is: tensor([28, 39, 42, 39, 44, 32, 49,  1]) target is: tensor(25)


In [36]:
class BigramLanguagemodel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)


    def forward(self, index, targets):
        logits = self.token_embedding_table(index)
        B,T,C = logits.shape  #T "time" is the sequence integer, C "channel" is vocab_size
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        
        return logits
        